# LeetCode #64 — Minimum Path Sum: DP tối ưu (min) trên lưới

**Mức độ:** Medium · **Chủ đề:** Array, Dynamic Programming, Matrix

Skeleton trên LeetCode:

```python
class Solution:
    def minPathSum(self, grid: List[List[int]]) -> int:
```

Hai bài trước (#62, #63) **đếm** số đường bằng phép **cộng**. Bài này **tối ưu**: tìm đường có tổng **nhỏ nhất**, bằng phép **min**. Khung DP giữ nguyên, chỉ đổi phép toán — một bài học quan trọng về tính tổng quát của DP.

Mục tiêu của notebook:

1. Công thức `dp[r][c] = grid[r][c] + min(trên, trái)`.
2. Vì sao **greedy** (luôn chọn ô kế tiếp nhỏ hơn) sai.
3. Bảng DP, một hàng, và sửa trực tiếp trên `grid`.
4. **Truy vết** để lấy chính đường đi tối ưu.

# 1. Bài toán là gì?

Cho lưới `m × n` gồm số **không âm**. Tìm đường từ góc trên trái tới góc dưới phải (chỉ đi **phải** hoặc **xuống**) sao cho **tổng các số trên đường** nhỏ nhất. Trả về tổng đó.

**Constraints (theo đề):**

- `1 <= m, n <= 200`
- `0 <= grid[i][j] <= 200`

---

## Ví dụ

```text
1 3 1
1 5 1      ->  7     (1 → 3 → 1 → 1 → 1)
4 2 1

1 2 3
4 5 6      ->  12    (1 → 2 → 3 → 6)
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Quy hoạch động (Dynamic Programming — DP)
2. DP hai chiều (bảng `dp[i][j]`)

Cuối mục có phần **2.3 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Quy hoạch động (Dynamic Programming — DP)

### Khi nào dùng DP?

Quy hoạch động dùng cho bài toán có hai đặc điểm:

1. **Cấu trúc con tối ưu**: đáp án của bài lớn được ghép từ đáp án của các **bài con** nhỏ hơn.
2. **Bài con chồng lấp**: cùng một bài con xuất hiện **nhiều lần** nếu đệ quy thẳng.

Thay vì tính lại, ta **lưu** đáp án mỗi bài con vào bảng và dùng lại → mỗi bài con tính đúng một lần.

### Bốn câu hỏi khi thiết kế DP

1. **Trạng thái**: `dp[i]` nghĩa là gì? (viết thành câu tiếng Việt rõ ràng)
2. **Chuyển trạng thái**: `dp[i]` tính từ những `dp[j]` nào?
3. **Cơ sở**: giá trị nhỏ nhất biết sẵn (`dp[0]`, `dp[1]`…).
4. **Đáp án**: là `dp[n]`, hay `max(dp)`, hay …?

### Ví dụ: leo cầu thang

Mỗi bước leo 1 hoặc 2 bậc. Có bao nhiêu cách leo `n` bậc?

- Trạng thái: `dp[i]` = số cách lên tới bậc `i`.
- Chuyển: bước cuối cùng là 1 bậc (từ `i−1`) hoặc 2 bậc (từ `i−2`) → `dp[i] = dp[i−1] + dp[i−2]`.
- Cơ sở: `dp[0] = 1` (đứng yên), `dp[1] = 1`.

In [1]:
from functools import lru_cache

# Cách 1: đệ quy thuần — chậm vì tính lặp
def leo_de_quy(n):
    return 1 if n <= 1 else leo_de_quy(n - 1) + leo_de_quy(n - 2)

# Cách 2: đệ quy + ghi nhớ (top-down, memoization)
@lru_cache(maxsize=None)
def leo_memo(n):
    return 1 if n <= 1 else leo_memo(n - 1) + leo_memo(n - 2)

# Cách 3: điền bảng từ nhỏ tới lớn (bottom-up, tabulation)
def leo_bang(n):
    dp = [0] * (n + 1)
    dp[0] = 1
    if n >= 1:
        dp[1] = 1
    for i in range(2, n + 1):
        dp[i] = dp[i - 1] + dp[i - 2]
    return dp[n], dp

# Cách 4: chỉ giữ hai giá trị gần nhất — O(1) bộ nhớ
def leo_gon(n):
    a, b = 1, 1                 # dp[i-2], dp[i-1]
    for _ in range(2, n + 1):
        a, b = b, a + b
    return b

print("n = 10:", leo_de_quy(10), leo_memo(10), leo_bang(10)[0], leo_gon(10))
print("bảng dp:", leo_bang(10)[1])

n = 10: 89 89 89 89
bảng dp: [1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89]


### Top-down hay bottom-up?

| | Top-down (memo) | Bottom-up (bảng) |
|---|---|---|
| Viết | gần với công thức đệ quy, dễ nghĩ | phải chọn **thứ tự** điền bảng |
| Chỉ tính bài con cần thiết | có | không (tính hết) |
| Rủi ro | độ sâu đệ quy | không |
| Tối ưu bộ nhớ | khó | dễ (giữ vài hàng/ô gần nhất) |

### Ví dụ thứ hai: chọn hay không chọn (House Robber)

Dãy nhà có tiền `a[i]`, không được lấy hai nhà **liền nhau**. Lấy tối đa bao nhiêu?

- `dp[i]` = số tiền tối đa khi chỉ xét `i` nhà đầu.
- Nhà thứ `i` (chỉ số `i−1`): **không lấy** → `dp[i−1]`; **lấy** → `dp[i−2] + a[i−1]`. Lấy `max`.

In [2]:
def ke_trom(a):
    dp = [0] * (len(a) + 1)
    if a:
        dp[1] = a[0]
    for i in range(2, len(a) + 1):
        dp[i] = max(dp[i - 1], dp[i - 2] + a[i - 1])
    return dp[-1], dp

print("ke_trom([2, 7, 9, 3, 1]) =", ke_trom([2, 7, 9, 3, 1]))

ke_trom([2, 7, 9, 3, 1]) = (12, [0, 2, 7, 11, 11, 12])


### Lỗi hay gặp

- Định nghĩa trạng thái mơ hồ → công thức sai. Luôn viết `dp[i] = …` bằng lời trước.
- Lệch chỉ số giữa `dp` (dài `n+1`) và mảng đầu vào (dài `n`).
- Điền bảng sai thứ tự (dùng ô chưa tính).
- Quên cơ sở hoặc khởi tạo sai (`0` thay vì `-∞` cho bài tìm max có giá trị âm…).
- Tối ưu bộ nhớ quá sớm khi chưa chắc công thức đúng — hãy viết bản bảng đầy đủ trước.

## 2.2 DP hai chiều (bảng `dp[i][j]`)

### Khi nào cần bảng hai chiều?

Khi trạng thái cần **hai** thông số, thường gặp ở:

- **Lưới**: `dp[i][j]` = đáp án cho ô `(i, j)` (số đường đi, tổng nhỏ nhất…).
- **Hai chuỗi**: `dp[i][j]` = đáp án cho **tiền tố** `s[:i]` và `t[:j]` (edit distance, LCS, xen kẽ…).
- **Đoạn con**: `dp[i][j]` = đáp án cho đoạn `s[i..j]` (palindrome…).

### Ví dụ 1: số đường đi trên lưới

Robot ở góc trên-trái, mỗi bước đi **phải** hoặc **xuống**. Có bao nhiêu đường tới ô `(i, j)`? Bước cuối đến từ ô **trên** hoặc ô **trái**:

```text
dp[i][j] = dp[i-1][j] + dp[i][j-1]
hàng 0 và cột 0 đều = 1 (chỉ một cách: đi thẳng)
```

In [3]:
def so_duong(m, n):
    dp = [[0] * n for _ in range(m)]      # KHÔNG dùng [[0] * n] * m
    for i in range(m):
        for j in range(n):
            if i == 0 or j == 0:
                dp[i][j] = 1
            else:
                dp[i][j] = dp[i - 1][j] + dp[i][j - 1]
    return dp

for row in so_duong(3, 5):
    print(" ".join(f"{v:>3}" for v in row))

  1   1   1   1   1
  1   2   3   4   5
  1   3   6  10  15


Thứ tự điền: theo hàng từ trên xuống, trong hàng từ trái sang → khi tính `dp[i][j]`, ô trên và ô trái **đã có**.

**Tiết kiệm bộ nhớ:** hàng `i` chỉ cần hàng `i−1` → giữ một mảng một chiều `row`, cập nhật `row[j] += row[j-1]` (giá trị cũ của `row[j]` chính là ô trên).

In [4]:
def so_duong_mot_hang(m, n):
    row = [1] * n
    for _ in range(1, m):
        for j in range(1, n):
            row[j] += row[j - 1]          # row[j] (cũ) = trên, row[j-1] (mới) = trái
    return row[-1]
print("so_duong_mot_hang(3, 7) =", so_duong_mot_hang(3, 7), "| so_duong(3, 7) =", so_duong(3, 7)[-1][-1])

so_duong_mot_hang(3, 7) = 28 | so_duong(3, 7) = 28


### Ví dụ 2: hai chuỗi — dãy con chung dài nhất (LCS)

`dp[i][j]` = độ dài dãy con chung dài nhất của `s[:i]` và `t[:j]`. Xét ký tự **cuối** của hai tiền tố:

- `s[i-1] == t[j-1]` → dùng nó: `dp[i][j] = dp[i-1][j-1] + 1`.
- khác → bỏ ký tự cuối của một trong hai: `dp[i][j] = max(dp[i-1][j], dp[i][j-1])`.

Hàng 0 và cột 0 ứng với **chuỗi rỗng** → bằng 0. Đó là lý do bảng có kích thước `(n+1) × (m+1)`.

In [5]:
def lcs(s, t):
    n, m = len(s), len(t)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if s[i - 1] == t[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
    return dp

s_demo, t_demo = "abcde", "ace"
bang = lcs(s_demo, t_demo)
print("      ''  " + "   ".join(t_demo))
for i, row in enumerate(bang):
    print(f"{('' if i == 0 else s_demo[i - 1])!r:>4} " + " ".join(f"{v:>3}" for v in row))
print("LCS =", bang[-1][-1])

      ''  a   c   e
  ''   0   0   0   0
 'a'   0   1   1   1
 'b'   0   1   1   1
 'c'   0   1   2   2
 'd'   0   1   2   2
 'e'   0   1   2   3
LCS = 3


### Ví dụ 3: khoảng cách sửa (edit distance) — số phép chèn/xóa/thay ít nhất

`dp[i][j]` = số phép sửa ít nhất biến `s[:i]` thành `t[:j]`.

- Ký tự cuối bằng nhau → `dp[i-1][j-1]`.
- Khác → `1 + min(thay: dp[i-1][j-1], xóa s[i-1]: dp[i-1][j], chèn t[j-1]: dp[i][j-1])`.
- Cơ sở: `dp[i][0] = i` (xóa hết), `dp[0][j] = j` (chèn hết).

In [6]:
def khoang_cach_sua(s, t):
    n, m = len(s), len(t)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1): dp[i][0] = i
    for j in range(m + 1): dp[0][j] = j
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if s[i - 1] == t[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(dp[i - 1][j - 1], dp[i - 1][j], dp[i][j - 1])
    return dp[n][m]

for a_, b_ in [("horse", "ros"), ("kitten", "sitting"), ("", "abc")]:
    print(f"{a_!r} → {b_!r}: {khoang_cach_sua(a_, b_)}")

'horse' → 'ros': 3
'kitten' → 'sitting': 3
'' → 'abc': 3


### Lỗi hay gặp

- `[[0] * n] * m` tạo `m` hàng là **cùng một** list → sửa một ô thì cả cột đổi.
- Lệch chỉ số: bảng `(n+1)×(m+1)` nhưng ký tự là `s[i-1]`, `t[j-1]`.
- Quên hàng/cột cơ sở (chuỗi rỗng, biên lưới).
- Khi tối ưu một hàng: ghi đè giá trị cần dùng lại (ví dụ `dp[i-1][j-1]`) — phải lưu tạm trước.

In [7]:
sai = [[0] * 3] * 2
sai[0][0] = 9
dung = [[0] * 3 for _ in range(2)]
dung[0][0] = 9
print("[[0]*3]*2 sau khi gán ô (0,0):", sai, "| tạo bằng vòng for:", dung)

[[0]*3]*2 sau khi gán ô (0,0): [[9, 0, 0], [9, 0, 0]] | tạo bằng vòng for: [[9, 0, 0], [0, 0, 0]]


## 2.3 Áp dụng vào bài này

**DP trên lưới** với phép `min` thay cho phép cộng: `dp[i][j] = grid[i][j] + min(dp[i−1][j], dp[i][j−1])`. Đi tham lam (chọn ô kề nhỏ hơn) **sai** — cần xét mọi đường, DP làm việc đó trong `O(mn)`.

# 3. Greedy có đúng không?

Ý tưởng dễ nghĩ: ở mỗi bước, đi sang ô (phải hoặc xuống) có giá trị **nhỏ hơn**.

In [8]:
from typing import List

def greedy_path_sum(grid: List[List[int]]) -> int:
    m, n = len(grid), len(grid[0])
    r = c = 0
    total = grid[0][0]
    while (r, c) != (m - 1, n - 1):
        if r == m - 1:
            c += 1
        elif c == n - 1:
            r += 1
        elif grid[r][c + 1] <= grid[r + 1][c]:
            c += 1
        else:
            r += 1
        total += grid[r][c]
    return total

print(greedy_path_sum([[1, 3, 1], [1, 5, 1], [4, 2, 1]]), "(đáp án đúng 7)")

9 (đáp án đúng 7)


Ngay ví dụ 1, greedy đã sai: từ `(0, 0)` nó chọn đi xuống (1 < 3), rồi lại xuống (4 < 5), và bị buộc đi qua số `4` — tổng `1 + 1 + 4 + 2 + 1 = 9`. Một ví dụ khác rõ hơn:

In [9]:
g = [[1, 1, 99],
     [9, 99, 99],
     [1, 1, 1]]
print("Greedy:", greedy_path_sum(g), "| Tối ưu: 1 + 9 + 1 + 1 + 1 = 13")

Greedy: 201 | Tối ưu: 1 + 9 + 1 + 1 + 1 = 13


Greedy đi sang phải theo số `1` rẻ, rồi kẹt giữa các số `99`. Đường tối ưu chấp nhận số `9` đắt hơn ở bước đầu để đi xuống hàng dưới rẻ.

> Khi lựa chọn hiện tại ảnh hưởng tới các lựa chọn sau, hãy nghĩ tới **DP**.

# 4. Công thức DP

`dp[r][c]` = tổng nhỏ nhất của đường từ `(0, 0)` tới `(r, c)`.

Bước cuối tới `(r, c)` từ trên hoặc từ trái → lấy cái **rẻ hơn**:

```text
dp[0][0] = grid[0][0]
hàng 0:  dp[0][c] = dp[0][c-1] + grid[0][c]     (chỉ có thể từ trái)
cột 0:   dp[r][0] = dp[r-1][0] + grid[r][0]     (chỉ có thể từ trên)
còn lại: dp[r][c] = grid[r][c] + min(dp[r-1][c], dp[r][c-1])
```

So sánh với #62: thay `+` (đếm) bằng `min` (tối ưu), và cộng thêm "chi phí" của ô.

In [10]:
def min_path_sum_table(grid: List[List[int]]) -> int:
    m, n = len(grid), len(grid[0])
    dp = [[0] * n for _ in range(m)]
    for r in range(m):
        for c in range(n):
            if r == 0 and c == 0:
                dp[r][c] = grid[0][0]
            elif r == 0:
                dp[r][c] = dp[r][c - 1] + grid[r][c]
            elif c == 0:
                dp[r][c] = dp[r - 1][c] + grid[r][c]
            else:
                dp[r][c] = grid[r][c] + min(dp[r - 1][c], dp[r][c - 1])
    return dp[m - 1][n - 1]

print(min_path_sum_table([[1, 3, 1], [1, 5, 1], [4, 2, 1]]))
print(min_path_sum_table([[1, 2, 3], [4, 5, 6]]))
print(min_path_sum_table(g))

7
12
13


# 5. Mô phỏng: bảng DP

```text
grid:          dp:
1 3 1          1 4 5
1 5 1          2 7 6
4 2 1          6 8 7
```

Ví dụ `dp[1][1] = 5 + min(dp[0][1]=4, dp[1][0]=2) = 7`; `dp[2][2] = 1 + min(6, 8) = 7`.

In [11]:
def show_dp(grid: List[List[int]]):
    m, n = len(grid), len(grid[0])
    dp = [[0] * n for _ in range(m)]
    for r in range(m):
        for c in range(n):
            if r == 0 and c == 0:
                dp[r][c] = grid[0][0]
            elif r == 0:
                dp[r][c] = dp[r][c - 1] + grid[r][c]
            elif c == 0:
                dp[r][c] = dp[r - 1][c] + grid[r][c]
            else:
                dp[r][c] = grid[r][c] + min(dp[r - 1][c], dp[r][c - 1])
    w = max(len(str(x)) for row in dp + grid for x in row)
    print("grid".ljust(n * (w + 1) + 3), "dp")
    for r in range(m):
        print(" ".join(str(x).rjust(w) for x in grid[r]), "  ", " ".join(str(x).rjust(w) for x in dp[r]))
    return dp

_ = show_dp([[1, 3, 1], [1, 5, 1], [4, 2, 1]])

grid      dp
1 3 1    1 4 5
1 5 1    2 7 6
4 2 1    6 8 7


# 6. Tối ưu bộ nhớ

**Một hàng:** giống #62, `row[c]` là hàng trên trước khi cập nhật, `row[c - 1]` là hàng hiện tại.

**Sửa tại chỗ `grid`:** dùng chính `grid` làm bảng DP — bộ nhớ `O(1)`, nhưng **phá hủy input** (chỉ nên làm khi được phép).

In [12]:
def min_path_sum_1d(grid: List[List[int]]) -> int:
    n = len(grid[0])
    row = [float("inf")] * n
    row[0] = 0
    for grid_row in grid:
        row[0] += grid_row[0]
        for c in range(1, n):
            row[c] = grid_row[c] + min(row[c], row[c - 1])
    return row[-1]


def min_path_sum_inplace(grid: List[List[int]]) -> int:
    m, n = len(grid), len(grid[0])
    for r in range(m):
        for c in range(n):
            if r == 0 and c == 0:
                continue
            top = grid[r - 1][c] if r > 0 else float("inf")
            left = grid[r][c - 1] if c > 0 else float("inf")
            grid[r][c] += min(top, left)
    return grid[-1][-1]

print(min_path_sum_1d([[1, 3, 1], [1, 5, 1], [4, 2, 1]]))
print(min_path_sum_inplace([[1, 3, 1], [1, 5, 1], [4, 2, 1]]))

7
7


Mẹo `float("inf")`: khởi tạo hàng "phía trên hàng 0" bằng vô cùng, để `min` tự chọn ô bên trái ở hàng 0. Tương tự, ô ngoài lưới coi như `inf`.

# 7. Phiên bản đúng format LeetCode

Ta dùng bản một hàng để không sửa input.

In [13]:
class Solution:
    def minPathSum(self, grid: List[List[int]]) -> int:
        n = len(grid[0])
        row = [float("inf")] * n
        row[0] = 0
        for grid_row in grid:
            row[0] += grid_row[0]
            for c in range(1, n):
                row[c] = grid_row[c] + min(row[c], row[c - 1])
        return row[-1]

In [14]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<20} PASS  {actual}")


def brute_min(grid, r=0, c=0):
    m, n = len(grid), len(grid[0])
    if r == m - 1 and c == n - 1:
        return grid[r][c]
    options = []
    if r + 1 < m:
        options.append(brute_min(grid, r + 1, c))
    if c + 1 < n:
        options.append(brute_min(grid, r, c + 1))
    return grid[r][c] + min(options)


sol = Solution()
check("Ví dụ 1", sol.minPathSum([[1, 3, 1], [1, 5, 1], [4, 2, 1]]), 7)
check("Ví dụ 2", sol.minPathSum([[1, 2, 3], [4, 5, 6]]), 12)
check("1 x 1", sol.minPathSum([[5]]), 5)
check("Một hàng", sol.minPathSum([[1, 2, 3]]), 6)
check("Một cột", sol.minPathSum([[1], [2], [3]]), 6)
check("Greedy bị lừa", sol.minPathSum(g), 13)
check("Toàn 0", sol.minPathSum([[0, 0], [0, 0]]), 0)

import random
random.seed(64)
for _ in range(2000):
    m, n = random.randint(1, 6), random.randint(1, 6)
    grid = [[random.randint(0, 20) for _ in range(n)] for _ in range(m)]
    expected = brute_min(grid)
    assert sol.minPathSum(grid) == expected
    assert min_path_sum_table(grid) == expected
    assert min_path_sum_inplace([row[:] for row in grid]) == expected
print("2000 lưới ngẫu nhiên: ba bản DP khớp brute force")

Ví dụ 1              PASS  7
Ví dụ 2              PASS  12
1 x 1                PASS  5
Một hàng             PASS  6
Một cột              PASS  6
Greedy bị lừa        PASS  13
Toàn 0               PASS  0


2000 lưới ngẫu nhiên: ba bản DP khớp brute force

# 8. Vì sao đúng? Độ phức tạp?

**Cấu trúc con tối ưu:** nếu đường tốt nhất tới `(r, c)` đi qua ô trước `p`, thì đoạn từ `(0, 0)` tới `p` **cũng phải** là đường tốt nhất tới `p` (nếu không, thay bằng đường tốt hơn tới `p` sẽ được đường tốt hơn tới `(r, c)`). Vì vậy `dp[r][c] = grid[r][c] + min(dp[trên], dp[trái])`.

Greedy sai vì nó chỉ nhìn **một bước**; DP xét **mọi** khả năng thông qua bảng con.

```text
Time:  O(m · n)
Space: O(n) (một hàng) | O(1) nếu sửa trên grid
```

# 9. Những lỗi dễ mắc

## Lỗi 1: Khởi tạo hàng 0 / cột 0 bằng `grid` thay vì tổng cộng dồn

Ô `(0, 2)` phải là `grid[0][0] + grid[0][1] + grid[0][2]`, không phải `grid[0][2]`.

In [15]:
def wrong_first_row(grid: List[List[int]]) -> int:
    m, n = len(grid), len(grid[0])
    dp = [row[:] for row in grid]                   # SAI: hàng 0, cột 0 chưa cộng dồn
    for r in range(1, m):
        for c in range(1, n):
            dp[r][c] = grid[r][c] + min(dp[r - 1][c], dp[r][c - 1])
    return dp[-1][-1]

print("Sai :", wrong_first_row([[1, 3, 1], [1, 5, 1], [4, 2, 1]]), "| Đúng:", sol.minPathSum([[1, 3, 1], [1, 5, 1], [4, 2, 1]]))

Sai : 3 | Đúng: 7


## Lỗi 2: Khởi tạo `row` bằng 0 thay vì `inf`

Với bản một hàng, hàng "ảo" phía trên phải là `inf`; nếu là 0, `min(0, ...)` sẽ coi như có đường miễn phí từ ngoài lưới vào.

In [16]:
def wrong_zero_init(grid: List[List[int]]) -> int:
    n = len(grid[0])
    row = [0] * n                                   # SAI
    for grid_row in grid:
        row[0] += grid_row[0]
        for c in range(1, n):
            row[c] = grid_row[c] + min(row[c], row[c - 1])
    return row[-1]

print("Sai :", wrong_zero_init([[1, 3, 1], [1, 5, 1], [4, 2, 1]]), "| Đúng: 7")

Sai : 3 | Đúng: 7


## Lỗi 3: Dùng greedy

Như mục 3.

## Lỗi 4: Quên cộng ô xuất phát hoặc ô đích

Tổng đường đi bao gồm **cả hai đầu**.

# 10. Mẹo nhận diện pattern

Khung DP trên lưới giống nhau, chỉ thay "phép gộp":

| Bài | Phép gộp | Giá trị ô cấm |
|---|---|---|
| #62 Unique Paths | `+` | — |
| #63 Unique Paths II | `+` | 0 |
| #64 Minimum Path Sum | `min` + chi phí ô | `inf` |
| "Có tới được không" | `or` | False |
| Đường có tổng **lớn nhất** | `max` + giá trị ô | `-inf` |

Bài liên quan: **#120 Triangle**, **#931 Minimum Falling Path Sum**, **#174 Dungeon Game**, **#741 Cherry Pickup** (Hard).

# 11. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Lập bảng DP cho

```text
1 2 5
3 2 1
```

## Bài 2 — Điền chỗ trống (bảng DP)

In [17]:
def exercise_2_min_path_sum(grid: List[List[int]]) -> int:
    m, n = len(grid), len(grid[0])
    dp = [[0] * n for _ in range(m)]
    # TODO: điền dp với 4 trường hợp: ô đầu, hàng 0, cột 0, còn lại
    return dp[m - 1][n - 1]


# assert exercise_2_min_path_sum([[1, 3, 1], [1, 5, 1], [4, 2, 1]]) == 7
# assert exercise_2_min_path_sum([[1, 2, 3], [4, 5, 6]]) == 12
# assert exercise_2_min_path_sum([[5]]) == 5

## Bài 3 — Truy vết đường đi

Trả về `(tổng, đường_đi)`, với `đường_đi` là list tọa độ từ `(0, 0)` tới `(m-1, n-1)`. Nếu hai hướng bằng nhau, ưu tiên đi từ **trên** xuống.

```python
min_path([[1, 3, 1], [1, 5, 1], [4, 2, 1]]) -> (7, [(0,0), (0,1), (0,2), (1,2), (2,2)])
```

Gợi ý nhanh: dựng bảng `dp` đầy đủ, rồi đi **ngược** từ đích: ở mỗi ô, bước trước là ô (trên hoặc trái) có `dp` nhỏ hơn.

In [18]:
def exercise_3_min_path(grid: List[List[int]]):
    # TODO
    pass


# assert exercise_3_min_path([[1, 3, 1], [1, 5, 1], [4, 2, 1]]) == (7, [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2)])
# assert exercise_3_min_path([[5]]) == (5, [(0, 0)])

## Bài 4 — Biến thể: Triangle (LeetCode #120)

Tam giác số; từ mỗi số ở hàng `i`, vị trí `j`, được đi xuống `j` hoặc `j + 1` của hàng `i + 1`. Tìm tổng nhỏ nhất từ đỉnh tới đáy.

```python
minimum_total([[2], [3, 4], [6, 5, 7], [4, 1, 8, 3]]) -> 11     # 2 + 3 + 5 + 1
```

Gợi ý nhanh: DP **từ đáy lên** — không cần xử lý biên.

In [19]:
def exercise_4_minimum_total(triangle: List[List[int]]) -> int:
    # TODO
    pass


# assert exercise_4_minimum_total([[2], [3, 4], [6, 5, 7], [4, 1, 8, 3]]) == 11
# assert exercise_4_minimum_total([[-10]]) == -10

## Bài 5 — Mini challenge: tổng lớn nhất khi được đi thêm sang trái

Lưới số không âm; được đi **phải**, **xuống**, hoặc **sang trái** (không đi lên), không đi lại ô đã đi. Tìm tổng **lớn nhất** từ hàng 0 (ô bất kỳ) tới hàng cuối (ô bất kỳ).

Gợi ý nhanh: trong mỗi hàng, đường đi là "vào ở cột `a`, đi ngang tới cột `b`, rồi xuống". Với mỗi hàng, tính hai lượt: quét trái → phải và phải → trái.

```python
max_snake([[1, 2], [3, 4]])  -> 10   # 1 → 2 → 4 → 3
```

In [20]:
def exercise_5_max_snake(grid: List[List[int]]) -> int:
    # TODO
    pass


# assert exercise_5_max_snake([[1, 2], [3, 4]]) == 10
# assert exercise_5_max_snake([[5]]) == 5
# assert exercise_5_max_snake([[1, 1, 1]]) == 3

# 12. Gợi ý

**Bài 1**
- Mức 1: hàng 0 của `dp`: `1, 3, 8`; cột 0: `1, 4`.

**Bài 3**
- Mức 1: bắt đầu `r, c = m-1, n-1`; lặp tới `(0, 0)`.
- Mức 2: nếu `r == 0` thì đi trái; nếu `c == 0` thì đi lên; ngược lại so `dp[r-1][c] <= dp[r][c-1]`.
- Mức 3: thêm tọa độ vào list rồi đảo ngược ở cuối.

**Bài 4**
- Mức 1: `dp = triangle[-1][:]`; đi từ hàng áp chót lên: `dp[j] = tri[i][j] + min(dp[j], dp[j+1])`.

**Bài 5**
- Mức 1: `best[c]` = tổng lớn nhất kết thúc tại hàng hiện tại, cột `c`, sau khi đã đi ngang xong.
- Mức 2: với hàng mới: `down[c] = best_prev[c] + g[c]` (vào thẳng từ trên); rồi `L[c] = max(down[c], L[c-1] + g[c])` (đi sang phải) và `R[c] = max(down[c], R[c+1] + g[c])` (đi sang trái); `best[c] = max(L[c], R[c])`.
- Mức 3: hàng đầu: `best_prev = [0] * n` (được bắt đầu ở ô bất kỳ).

# 13. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [21]:
_ = show_dp([[1, 2, 5], [3, 2, 1]])

grid      dp
1 2 5    1 3 8
3 2 1    4 5 6


## Đáp án Bài 2

In [22]:
def answer_2_min_path_sum(grid: List[List[int]]) -> int:
    return min_path_sum_table(grid)

assert answer_2_min_path_sum([[1, 3, 1], [1, 5, 1], [4, 2, 1]]) == 7
assert answer_2_min_path_sum([[1, 2, 3], [4, 5, 6]]) == 12
assert answer_2_min_path_sum([[5]]) == 5
print("Bài 2 PASS (giống min_path_sum_table ở mục 4)")

Bài 2 PASS (giống min_path_sum_table ở mục 4)


## Đáp án Bài 3

In [23]:
def answer_3_min_path(grid: List[List[int]]):
    m, n = len(grid), len(grid[0])
    dp = [[0] * n for _ in range(m)]
    for r in range(m):
        for c in range(n):
            if r == 0 and c == 0:
                dp[r][c] = grid[0][0]
            elif r == 0:
                dp[r][c] = dp[r][c - 1] + grid[r][c]
            elif c == 0:
                dp[r][c] = dp[r - 1][c] + grid[r][c]
            else:
                dp[r][c] = grid[r][c] + min(dp[r - 1][c], dp[r][c - 1])
    path = []
    r, c = m - 1, n - 1
    while True:
        path.append((r, c))
        if r == 0 and c == 0:
            break
        if r == 0:
            c -= 1
        elif c == 0:
            r -= 1
        elif dp[r - 1][c] <= dp[r][c - 1]:
            r -= 1
        else:
            c -= 1
    return dp[-1][-1], path[::-1]

assert answer_3_min_path([[1, 3, 1], [1, 5, 1], [4, 2, 1]]) == (7, [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2)])
assert answer_3_min_path([[5]]) == (5, [(0, 0)])
for _ in range(1000):
    m, n = random.randint(1, 6), random.randint(1, 6)
    grid = [[random.randint(0, 9) for _ in range(n)] for _ in range(m)]
    total, path = answer_3_min_path(grid)
    assert total == sum(grid[r][c] for r, c in path) == sol.minPathSum(grid)
    assert all((r2 - r1, c2 - c1) in ((1, 0), (0, 1)) for (r1, c1), (r2, c2) in zip(path, path[1:]))
print("Bài 3 PASS (đường đi hợp lệ và có tổng tối ưu trên 1000 test)")

Bài 3 PASS (đường đi hợp lệ và có tổng tối ưu trên 1000 test)

## Đáp án Bài 4

In [24]:
def answer_4_minimum_total(triangle: List[List[int]]) -> int:
    dp = triangle[-1][:]
    for i in range(len(triangle) - 2, -1, -1):
        for j in range(i + 1):
            dp[j] = triangle[i][j] + min(dp[j], dp[j + 1])
    return dp[0]

assert answer_4_minimum_total([[2], [3, 4], [6, 5, 7], [4, 1, 8, 3]]) == 11
assert answer_4_minimum_total([[-10]]) == -10
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [25]:
def answer_5_max_snake(grid: List[List[int]]) -> int:
    n = len(grid[0])
    best_prev = [0] * n
    for g_row in grid:
        down = [best_prev[c] + g_row[c] for c in range(n)]
        L = down[:]
        for c in range(1, n):
            L[c] = max(L[c], L[c - 1] + g_row[c])
        R = down[:]
        for c in range(n - 2, -1, -1):
            R[c] = max(R[c], R[c + 1] + g_row[c])
        best_prev = [max(L[c], R[c]) for c in range(n)]
    return max(best_prev)

def brute_snake(grid):
    m, n = len(grid), len(grid[0])
    best = 0
    def dfs(r, c, seen, total):
        nonlocal best
        if r == m - 1:
            best = max(best, total)
        for dr, dc in ((0, 1), (0, -1), (1, 0)):
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and (nr, nc) not in seen:
                seen.add((nr, nc)); dfs(nr, nc, seen, total + grid[nr][nc]); seen.remove((nr, nc))
    for c in range(n):
        dfs(0, c, {(0, c)}, grid[0][c])
    return best

assert answer_5_max_snake([[1, 2], [3, 4]]) == 10
assert answer_5_max_snake([[5]]) == 5
assert answer_5_max_snake([[1, 1, 1]]) == 3
for _ in range(500):
    m, n = random.randint(1, 4), random.randint(1, 4)
    grid = [[random.randint(0, 9) for _ in range(n)] for _ in range(m)]
    assert answer_5_max_snake(grid) == brute_snake(grid), grid
print("Bài 5 PASS (khớp brute force trên 500 lưới)")

Bài 5 PASS (khớp brute force trên 500 lưới)


Bài 5 cho thấy khi được đi **hai chiều** trong một hàng, ta không còn thứ tự điền đơn giản "trái sang phải" — phải tách thành hai lượt quét. Đây là kỹ thuật gặp ở nhiều bài DP trên lưới khó hơn.

# 14. Tổng kết

> `dp[r][c] = grid[r][c] + min(dp[trên], dp[trái])`. Khung giống #62, chỉ thay `+` bằng `min` và cộng chi phí ô. Greedy sai vì không nhìn xa; DP đúng nhờ **cấu trúc con tối ưu**.

---

## Checklist trước khi nộp

- [ ] Hàng 0 / cột 0 là tổng **cộng dồn**?
- [ ] Bản một hàng khởi tạo bằng `inf` (trừ `row[0] = 0`)?
- [ ] Có cộng cả ô đầu và ô đích?
- [ ] Không sửa `grid` nếu không được phép?